<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Joint%20Angle%20Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


This notebook serves as the central command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)
* **💻 Project Code:** [Wearable-Biomechanics Repository](https://github.com/janithcyapa/Wearable-Biomechanics/)
* **📂 Data Storage:** [Project Google Drive Folder](https://drive.google.com/drive/folders/1QERF02T4pvubAEjnv6f1VrCOWNYxHv54?usp=drive_link)


## 📚 Initial Setup & Configuration


### 🔧 One-Time Setup Instructions

#### **Step 1: Get Access to the Shared Dataset Folder**
1. Ask your **admin** to share the Google Drive folder with you
2. You'll receive an email with a "Shared with you" notification
3. Open the shared folder in Google Drive
4. The folder should be named: **`Wearable-Biomechanics`**

#### **Step 2: Add Shortcut to Your Drive**
1. In Google Drive, locate the shared folder under **"Shared with me"**
2. **Right-click** on the folder named `Wearable-Biomechanics`
3. Select **"Add shortcut to Drive"** (or **"Organize → Add shortcut"**)
4. Navigate to your **`My Drive > Datasets`** folder
   - If `Datasets` folder doesn't exist, create it first
5. Click **"Add Shortcut"**

#### **Step 3: Verify the Setup**
- Go to **`My Drive > Datasets`** in your Google Drive
- You should see **`Wearable-Biomechanics`** with a shortcut icon ⤴️
- The full path will be: **`MyDrive/Datasets/Wearable-Biomechanics`**


### ⚠️ Important Notes
- 🔑 **Permissions**: You need at least **"Editor"** access to upload data
- 🔗 **Shortcuts**: Always use the shortcut in your Drive, not the original shared folder
- 📂 **Structure**: The system will auto-create `/index`, `/raw_data`, and `/processed_data` subfolders
- 🚫 **Don't delete** the shortcut or you'll lose access to shared data


### 📁 Folder Structure After Setup
```
My Drive/
└── Datasets/
    └── Wearable-Biomechanics/ (shortcut ⤴️)
        ├── index/                  ## JSON indexes
        │   ├── entities_subjects.json
        │   └── entries_experiments.json
        ├── raw_data/               ## Uploaded CSV files
        │   └── experiments/
        └── processed_data/         ## Processed Parquet files
            └── experiments/
```


## **🛠️ Setup OpenDatasetStore**

In [ ]:

from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local", # Use "local" since you are mounting Google Drive via Colab directly
    entity_id_format="sub_{num:04d}", # You can drop the `f""` string formatting here now
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


Mounted at /content/drive
  Cloning https://github.com/janithcyapa/OpenDatasetStore.git to /tmp/pip-req-build-ybrc07d7
  Running command git clone --filter=blob:none --quiet https://github.com/janithcyapa/OpenDatasetStore.git /tmp/pip-req-build-ybrc07d7
  Resolved https://github.com/janithcyapa/OpenDatasetStore.git to commit 12827c3c1fafd1dd5c682793cb14470ece46ad69
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for open_dataset_store: filename=open_dataset_store-2.0.1-py3-none-any.whl size=24552 sha256=3398cd79e3e228a05bdc656297680d1688a96e1d28297b681341fcb11aaf227e
  Stored in directory: /tmp/pip-ephem-wheel-cache-wmm3w9yb/wheels/63/22/ef/d2f6ce9e67edffbf2ce93b9864b71460150ac2e5309ee86676
Successfully built open_dataset_store
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)


# **👤 People Management (Subject)**

In [ ]:
# @title 📝 Register New Subject { display-mode: "form" }
# @markdown Enter the details of the test subject to generate a unique ID.

name = "Pandula" # @param {type:"string"}
age = 00 # @param {type:"number"}
weight_kg = 00 # @param {type:"number"}
height_m = 00 # @param {type:"number"}
achievement = "NA"# @param {type:"string"}
best_weight = "00"# @param {type:"string"}
experience_years = 00 # @param {type:"integer"}
notes = "NA" # @param {type:"string"}

# Logic to create entity
sub_id = store.create_entity(
    entity_type="subjects",
    name=name,
    age=age,
    weight=weight_kg,
    height=height_m,
    experience=experience_years,
    achievement=achievement,
    best_weight=best_weight,
    notes=notes
)

print(f"✅ Created Subject ID: {sub_id}")

In [ ]:
# @title 📋 Participant Registry { display-mode: "form" }
# @markdown View and filter the registered test subjects.

from google.colab import data_table

# Fetch subject data
df_subjects = store.list_entities("subjects")

if not df_subjects.empty:
    # 1. Clean up column names for display (e.g., replace underscores with spaces)
    df_subjects.columns = [c.replace('_', ' ').title() for c in df_subjects.columns]

    # 2. Display using Colab's Interactive Data Table
    print(f"Total Subjects Registered: {len(df_subjects)}")
    display(data_table.DataTable(
        df_subjects,
        include_index=False,
        num_rows_per_page=10,
        max_columns=20
    ))
else:
    print("⚠️ No participants found in the registry.")

Total Subjects Registered: 10


,Entity Id,Name,Age,Weight,Height,Experience,Achievement,Best Weight,Notes
0,sub_0001,Hivindu Kirulapana,22,72,1.74,1,-,-,Participation in Other Sports - Rugby
1,sub_0002,Yasith Dissanayake,22,69,178.00,4,-,55,Participation in Other Sports - Rugby
2,sub_0003,Yewin Dissanayake,22,65,170.00,2,Intermediate bronze medal 2025,95,Participation in Other sports: none
3,sub_0004,Mahasen,0,0,0.00,0,N\A,00,N\A
4,sub_0005,Achalani,0,0,0.00,0,N\A,00,N\A
5,sub_0006,Pandula,0,0,0.00,0,NA,00,NA
6,sub_0007,Enuka,0,0,0.00,0,NA,00,NA
7,sub_0008,Thevindu,0,0,0.00,0,NA,00,NA
8,sub_0009,Induwara,0,0,0.00,0,NA,00,NA
9,sub_0010,Duleetha,0,0,0.00,0,NA,00,NA


# **📂 Experiment Upload & Data Exploration**

In [ ]:
# @title 🚀 Upload New Experiment { display-mode: "form" }
# @markdown Select a subject and upload their raw IMU data.

target_subject_id = "sub_0006" # @param {type:"string"}
activity_type = "Clean & Jerk" # @param {type:"string"}
weight_used = 20 # @param {type:"number"}
date = '2025-10-01' # @param {type:"string"}
time = '19:21' # @param {type:"string"}
timestamp = 1759326713 # @param {type:"number"}

notes = '5 reps, Complete' # @param {type:"string"}

exp_id = store.create_entry_interactive(
    entry_type="experiments",
    entity_id=target_subject_id,
    date=date,
    time=time,
    description=activity_type,
    timestamp=timestamp,
    weight_kg=weight_used,
    notes=notes
)

# @markdown ---
# @markdown ### 📈 Auto-Visualization
# @markdown After upload, the IMU data will be plotted below for quality check.

if exp_id:
    raw_df = store.get_entry_raw_data("experiments", exp_id)

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Dynamically find Accelerometer columns (assuming AX, AY, AZ naming)
    accel_cols = [c for c in raw_df.columns if '_AX' in c or '_AY' in c or '_AZ' in c][:3]

    fig = go.Figure()
    for col in accel_cols:
        fig.add_trace(go.Scatter(y=raw_df[col], name=col, mode='lines'))

    fig.update_layout(
        title=f"Initial Data Check: {exp_id}",
        xaxis_title="Samples",
        yaxis_title="Acceleration ($m/s^2$)",
        template="plotly_dark",
        height=400
    )
    fig.show()

In [ ]:
# @title 🧪 Experiment Log { display-mode: "form" }
# @markdown Browse all uploaded biomechanical data sessions.

from google.colab import data_table
data_table.enable_dataframe_formatter()

# Fetch experiment entries
df_experiments = store.list_entries("experiments")

if not df_experiments.empty:
    # Organize columns so the most important info is first
    cols_priority = ["entry_id", "entity_id", "timestamp", "description"]
    other_cols = [c for c in df_experiments.columns if c not in cols_priority]
    df_experiments = df_experiments[cols_priority + other_cols]

    df_experiments = df_experiments.sort_values(by="entry_id", ascending=False)

    # Format the table for a cleaner look
    df_experiments.columns = [c.replace('_', ' ').title() for c in df_experiments.columns]

    display(data_table.DataTable(
        df_experiments,
        include_index=False,
        num_rows_per_page=10
    ))
else:
    print("⚠️ No experiments recorded yet.")

,Entry Id,Entity Id,Timestamp,Description,Raw Csv Path,Processed Files,Date,Time,Weight Kg,Notes,Related Files
34,experiments_0035,sub_0013,1784896459,Clean & Jerk,raw_data/experiments/experiments_0035_sub_0013...,{},2026-07-24,18:04,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
33,experiments_0034,sub_0013,1784896277,Clean & Jerk,raw_data/experiments/experiments_0034_sub_0013...,{},2026-07-24,18:01,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
32,experiments_0033,sub_0012,1784894597,Clean & Jerk,raw_data/experiments/experiments_0033_sub_0012...,{},2026-07-24,17:33,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
31,experiments_0032,sub_0012,1784894335,Clean & Jerk,raw_data/experiments/experiments_0032_sub_0012...,{},2026-07-24,17:28,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
30,experiments_0031,sub_0011,1784284923,Clean & Jerk,raw_data/experiments/experiments_0031_sub_0011...,{},2026-07-17,16:12,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
29,experiments_0030,sub_0011,1784284557,Clean & Jerk,raw_data/experiments/experiments_0030_sub_0011...,{},2026-07-17,16:05,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
28,experiments_0029,sub_0003,1780495107,Clean & Jerk,raw_data/experiments/experiments_0029_sub_0003...,{},2026-06-03,19:28,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
27,experiments_0028,sub_0003,1780495958,Clean & Jerk,raw_data/experiments/experiments_0028_sub_0003...,{},2026-06-03,19:42,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
26,experiments_0027,sub_0003,1780496235,Clean & Jerk,raw_data/experiments/experiments_0027_sub_0003...,{},2026-06-03,19:47,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...
25,experiments_0026,sub_0002,1780497975,Clean & Jerk,raw_data/experiments/experiments_0026_sub_0002...,{},2026-06-03,20:16,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...


In [ ]:
# @title 📎 Attachment Manager { display-mode: "form" }
# @markdown Upload related files (videos, high-speed clips, or archives) to an experiment.

experiment_to_attach = "experiments_0035" # @param {type:"string"}
file_label = "side_video" # @param ["front_video", "side_video", "raw_zip", "other"]

rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=experiment_to_attach,
    label=file_label
)

📤 Select file for 'side_video' (Entry: experiments_0035)...


Saving 20260724_180421.mp4 to 20260724_180421.mp4
📎 Attached: experiments_0035_sub_0013_side_video.mp4


# **📊  Advanced IMU Analysis Plot**

In [ ]:
# @title 🔍 Detailed Signal Analysis { display-mode: "form" }
# @markdown Compare different IMU nodes on the body.

selected_exp = "experiments_0002" # @param {type:"string"}
sensor_node_A = "device1" # @param {type:"string"}
sensor_node_B = "device2" # @param {type:"string"}

df_analysis = store.get_entry_raw_data("experiments", selected_exp)

import plotly.express as px

# Filter for specific nodes
plot_cols = [c for c in df_analysis.columns if sensor_node_A in c or sensor_node_B in c]
plot_cols = [c for c in plot_cols if "IMU1_AX" in c] # Just showing X-axis for comparison

fig = px.line(df_analysis, y=plot_cols,
              title=f"Comparison: {sensor_node_A} vs {sensor_node_B}",
              labels={"value": "G-Force", "index": "Time (Samples)"},
              template="simple_white")

fig.show()